# 框架

```mermaid
graph TD
    %% ------------------- 样式定义 (Style Definitions) -------------------
    classDef interface fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
    classDef foundation fill:#D5F5E3,stroke:#287C3E,stroke-width:2px
    classDef writer fill:#FAD7A0,stroke:#333,stroke-width:2px
    classDef reader fill:#E8DAEF,stroke:#333,stroke-width:2px
    classDef build fill:#EEEEEE,stroke:#888,stroke-width:1px

    %% ------------------- 顶层容器 (Top Level Subgraph) -------------------
    subgraph WtDataStorageAD ["高性能数据存储模块 (WtDataStorageAD) - 基于 LMDB 和内存映射"]
        direction LR

        %% ------------------- 1. 外部接口 (External Interfaces) -------------------
        subgraph ExternalInterfaces ["外部接口 (定义于 Includes/)"]
            direction TB
            IDataWriter("IDataWriter.h<br/><b>[接口]</b> 实时数据写入器<br/><b>意义:</b> 定义数据写入规范, 是数据生产者的统一入口"):::interface
            IDataReader("IDataReader.h<br/><b>[接口]</b> 实时数据读取器<br/><b>意义:</b> 定义实盘策略读取数据的规范 (实时+历史)"):::interface
            IBtDtReader("IBtDtReader.h<br/><b>[接口]</b> 回测数据读取器<br/><b>意义:</b> 定义回测引擎读取数据的规范 (性能优先)"):::interface
            IRdmDtReader("IRdmDtReader.h<br/><b>[接口]</b> 随机数据读取器<br/><b>意义:</b> 定义分析工具按需读取数据的规范 (灵活性优先)"):::interface
        end

        %% ------------------- 2. 模块内部基础 (Internal Foundation) -------------------
        subgraph InternalFoundation ["模块内部基础 (Internal Foundation)"]
            direction TB
            DataDefine("DataDefineAD.h<br/><b>[数据结构]</b> 内存缓存定义<br/><b>作用:</b> 定义内存映射的实时缓存结构<br/>(RTTickCache, RTBarCache)<br/><b>用法:</b> WtDataWriterAD 写入, WtDataReaderAD 读取"):::foundation
            LMDBKeys("LMDBKeys.h<br/><b>[数据结构]</b> LMDB键定义<br/><b>作用:</b> 定义数据库键结构 (LMDBHftKey, LMDBBarKey)<br/>包含字节序转换 (reverseEndian), 确保时间排序正确"):::foundation
        end

        %% ------------------- 3. 核心组件 (Core Components) -------------------
        subgraph CoreComponents ["核心实现 (Implementations)"]
            direction TB

            %% 3.1 写入器 (Writer)
            subgraph Writer ["数据写入 (Producer)<br/>职责: 接收实时数据, 写入缓存和数据库, 合成K线"]
                direction TB
                Writer_h("WtDataWriterAD.h<br/><b>[定义]</b> 高级数据写入器 (AD)"):::writer
                Writer_cpp("WtDataWriterAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 接收实时Tick, 异步写入LMDB, 合成K线, 更新实时缓存"):::writer
            end

            %% 3.2 实时读取器 (Real-time Reader)
            subgraph RlReader ["实时读取 (Real-time Consumer)<br/>职责: 供实盘策略使用"]
                direction TB
                Reader_h("WtDataReaderAD.h<br/><b>[定义]</b> 实时数据读取器 (AD)"):::reader
                Reader_cpp("WtDataReaderAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 融合LMDB(历史)+内存缓存(当日)+内存(最新)三级数据"):::reader
            end
            
            %% 3.3 回测读取器 (Backtest Reader)
            subgraph BtReader ["回测读取 (Backtest Consumer)<br/>职责: 供回测引擎使用"]
                direction TB
                BtReader_h("WtBtDtReaderAD.h<br/><b>[定义]</b> 回测数据读取器 (AD)"):::reader
                BtReader_cpp("WtBtDtReaderAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 高速读取原始(raw)二进制数据块, 极致回测性能"):::reader
            end

            %% 3.4 随机读取器 (Random-access Reader)
            subgraph RdmReader ["随机读取 (Analysis Consumer)<br/>职责: 供数据分析工具使用"]
                direction TB
                RdmReader_h("WtRdmDtReaderAD.h<br/><b>[定义]</b> 随机数据读取器 (AD)"):::reader
                RdmReader_cpp("WtRdmDtReaderAD.cpp<br/><b>[实现]</b><br/><b>用法:</b> 按需(ByRange, ByCount)加载数据到内存"):::reader
            end
        end
    end

    %% ------------------- 4. 关系连接 (Relationships) -------------------

    %% 4.1 接口实现关系 (Implementation)
    IDataWriter -- "实现 (implements)" --> Writer_h
    IDataReader -- "实现 (implements)" --> Reader_h
    IBtDtReader -- "实现 (implements)" --> BtReader_h
    IRdmDtReader -- "实现 (implements)" --> RdmReader_h

    %% 4.2 头文件与实现文件 (h/cpp)
    Writer_h --> Writer_cpp
    Reader_h --> Reader_cpp
    BtReader_h --> BtReader_cpp
    RdmReader_h --> RdmReader_cpp

    %% 4.3 核心依赖关系 (Core Dependencies)
    Writer_h -- "依赖 (uses)" --> DataDefine
    Writer_h -- "依赖 (uses)" --> LMDBKeys
    
    Reader_h -- "依赖 (uses)" --> DataDefine
    Reader_h -- "依赖 (uses)" --> LMDBKeys
    
    BtReader_h -- "依赖 (uses)" --> LMDBKeys
    RdmReader_h -- "依赖 (uses)" --> LMDBKeys
    
    %% 4.4 逻辑数据流 (Logical Data Flow)
    Writer_cpp -- "写入 (Writes to)" --> DataDefine
    Writer_cpp -- "写入 (Writes to)" --> LMDBKeys
    Reader_cpp -- "读取 (Reads from)" --> DataDefine
    Reader_cpp -- "读取 (Reads from)" --> LMDBKeys
    BtReader_cpp -- "读取 (Reads from)" --> LMDBKeys
    RdmReader_cpp -- "读取 (Reads from)" --> LMDBKeys
```

# DataDefineAD.h

- **核心常量与版本控制**
  - `const char BLK_FLAG[] = "&^%$#@!\0"`
    - **作用**：8 字节的“魔数”（Magic Number），用作文件的指纹。
    - **意义**：当 `WtDataReaderAD` 加载一个缓存文件时，它会首先检查文件头部是否有这个 `BLK_FLAG`。如果匹配，则确认这是一个有效的数据缓存文件；如果不匹配，则认为文件已损坏或格式不正确，拒绝加载。
  - `#define BLOCK_VERSION_RAW 1`
    - **作用**：定义了缓存文件的数据格式版本号。
    - **意义**：`RAW` 表示文件中的数据（如 `WTSTickStruct`, `WTSBarStruct`）是以**原始二进制 (raw)** 格式存储的，没有经过压缩。这为未来的格式升级（如添加压缩版本）提供了兼容性基础。

- **数据块类型枚举 (`BlockType`)**
  ```cpp
  typedef enum tagBlockType
  {
    BT_RT_Cache = 4
  } BlockType;
  ```
  - **作用**：标识数据块中存储的是哪一种数据。
  - **意义**：这是此文件中唯一定义的类型，代表**“实时缓存” (Real-Time Cache)**。
  - **区别**：这与标准版 `DataDefine.h` 中的 `BT_HIS_Minute1` (历史K线)、`BT_HIS_Ticks` (历史Tick) 等类型是**完全不同**的。`WtDataStorageAD` 模块使用 LMDB 存储历史数据，因此它不需要历史数据块类型，它只需要这个实时缓存类型来管理当日的内存映射文件。

- **数据块头部结构**
  ```cpp
  typedef struct _BlockHeader
  {
    char		_blk_flag[FLAG_SIZE]; // 数据块标识魔数，用于验证数据块的有效性
    uint16_t	_type;  // 数据块类型，对应BlockType枚举值
    uint16_t	_version; // 数据块版本号，用于格式兼容性管理
  } BlockHeader;
  ```
  - **作用**：作为所有实时缓存文件的通用元信息，提供了识别和解析数据的基础。
  - **意义**：这是所有缓存文件的基础身份信息。
  ```cpp
  typedef struct _RTBlockHeader : BlockHeader
  {
    uint32_t _size; // 当前已使用的数据项数量
    uint32_t _capacity; // 数据块的最大容量（数据项数量）
  } RTBlockHeader;
  ```
  - **定义**：继承自 `BlockHeader`，并增加了 `uint32_t _size` (当前条数) 和 `uint32_t _capacity` (总容量) 字段。
  - **区别与意义**：这是 `WtDataStorageAD` 缓存设计的核心。
    - `_capacity`：在文件创建时，会预先分配 `_capacity` * `sizeof(Item)` 的巨大空间。
    - `_size`：在交易过程中，`WtDataWriterAD` 每写入一条新数据，只会将 `_size` 加 1，而**不需要扩展文件**。
    - 这种“预分配容量、递增大小”的机制，避免了在行情高峰期频繁进行昂贵的文件 I/O 和扩容操作，是实现高性能写入的关键。

- **具体数据缓存项 (`CacheItem`) 定义**
  ```cpp
  typedef struct _TickCacheItem
  {
    uint32_t		_date;  // 交易日期（格式：YYYYMMDD）
    WTSTickStruct	_tick;  // Tick行情数据结构
  } TickCacheItem;
  ```
  - **意义**：这是实时 Tick 缓存 (`cache_tick.dmb`) 中每一条记录的格式。它将 `WTSTickStruct`（包含完整的行情快照）与一个交易日期 `_date` 绑定在一起。
  ```cpp
  typedef struct _BarCacheItem
  {
    char			_exchg[16]; // 交易所代码，固定16字节长度
    char			_code[32];  // 合约代码，固定32字节长度
    WTSBarStruct	_bar; // K线数据结构
  } BarCacheItem;
  ```
  - **意义**：这是实时 K 线缓存 (`cache_m1.dmb`, `cache_m5.dmb` 等) 中每一条记录的格式。

- **实时缓存块实现 (Flexible Array Member)**
  - **作用**：将头部与数据项结合起来，定义了完整的内存映射文件布局。
  - **用法**：使用了 C 语言中的“柔性数组成员”技巧 (`_items[0]`)。这意味着头部和它后面的 `_items` 数组存储在一块**连续的内存**中。
  ```cpp
  typedef struct _RTTickCache : RTBlockHeader
  {
    TickCacheItem	_items[0];  // 变长数组，存储Tick缓存项数据
  } RTTickCache;
  ```
  - **定义**：`RTBlockHeader` 后面紧跟着 `TickCacheItem _items[0]`。
  - **意义**：定义了 `cache_tick.dmb` 文件的内存布局。
  ```cpp
  typedef struct _RTBarCache : RTBlockHeader
  {
    BarCacheItem	_items[0];  // 变长数组，存储K线缓存项数据
  } RTBarCache;
  ```
  - **定义**：`RTBlockHeader` 后面紧跟着 `BarCacheItem _items[0]`。
  - **意义**：定义了 `cache_m1.dmb`, `cache_m5.dmb`, `cache_d1.dmb` 等 K 线缓存文件的内存布局。

# LMDBKeys.h

# WtDataReaderAD.h/cpp

# WtDataWriterAD.h/cpp

# WtBtDtReaderAD.h/cpp

# WtRdmDtReaderAD.h/cpp